# 柵付き砂丘シミュレーション（定数スイープ）

定数を変えたインスタンスを複数生成し、**400 ステップ時点の結果描写のみ**を画像保存して、
どの定数値が適しているかを比較する。

可変にするのは以下の 5 定数のみ（それ以外は比較条件を揃えるため固定）。

- `L0` : 跳躍距離の基準値（兼 風上からの砂供給幅）
- `A2` : 跳躍量 q の勾配依存係数
- `b`  : 跳躍距離 L の勾配依存係数
- `q0` : 跳躍量の基準値
- `D`  : クリープ（転がり）で動く砂の割合

In [19]:
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go  # 3D 描画用

# 岩盤（これ以上は削れない）高さ
BEDROCK = -10.0

# クリープの 8 近傍方向（左上・上・右上・左・右・左下・下・右下）
DIRECTIONS = [
    (-1, 1), (0, 1), (1, 1),
    (-1, 0),         (1, 0),
    (-1, -1), (0, -1), (1, -1),
]


class DuneSimulation:
    """砂丘シミュレーションの 1 試行を表すクラス。

    可変 5 定数を引数で受け取り、その他の定数はキーワード引数のデフォルトで固定する。
    run() で nt_max ステップ進め、400 ステップ時点の描写を画像保存する。

    境界条件:
        x 方向（風向き）… 周期境界。右端を出た砂は左端へ再流入する
                          （勾配計算・跳躍・クリープの輸送すべてを周期化）。
        y 方向（横断方向）… 壁境界。外向きの輸送はブロックする。
    """

    def __init__(
        self,
        L0,
        A2,
        b,
        q0,
        D,
        *,
        nx=110,
        ny=50,
        fence_x=60,
        fence_height=20,
        mu=0.2,
        g=9.8,
        M=1.0,
        a_init=1.5,
        seed=0,
        nt_max=400,
    ):
        # --- 可変 5 定数 ---
        self.L0 = L0
        self.A2 = A2
        self.b = b
        self.q0 = q0
        self.D = D

        # --- 固定定数（比較条件を揃えるためデフォルト固定） ---
        self.nx = nx
        self.ny = ny
        self.fence_x = fence_x
        self.fence_height = fence_height
        self.mu = mu
        self.g = g
        self.M = M
        self.a_init = a_init
        self.seed = seed
        self.nt_max = nt_max

        # --- 座標・初期地形 ---
        self.x = np.linspace(0.5, nx - 0.5, nx)
        self.y = np.linspace(0.5, ny - 0.5, ny)
        self.h = self._make_initial_field()

    # ------------------------------------------------------------------
    # 初期化
    # ------------------------------------------------------------------
    def _make_initial_field(self):
        """平均 0 の微小ランダム地形を生成する。"""
        rng = np.random.default_rng(self.seed)
        h = rng.uniform(-0.5 * self.a_init, 0.5 * self.a_init, size=(self.ny, self.nx))
        return h - np.mean(h)

    # ------------------------------------------------------------------
    # 物理ステップ
    # ------------------------------------------------------------------
    def _calc_saltation_flux(self, H, dx=1.0, dy=1.0):
        """Sobel フィルタで勾配を求め、跳躍量 q と跳躍距離 L を計算する。

        np.roll による差分は x・y いずれも周期境界として勾配を評価する。
        """
        H_R = np.roll(H, shift=-1, axis=1)
        H_L = np.roll(H, shift=1, axis=1)
        H_U = np.roll(H, shift=-1, axis=0)
        H_D = np.roll(H, shift=1, axis=0)

        H_RU = np.roll(H_R, shift=-1, axis=0)
        H_RD = np.roll(H_R, shift=1, axis=0)
        H_LU = np.roll(H_L, shift=-1, axis=0)
        H_LD = np.roll(H_L, shift=1, axis=0)

        dH_dx = ((H_RU + 2.0 * H_R + H_RD) - (H_LU + 2.0 * H_L + H_LD)) / (8.0 * dx)
        dH_dy = ((H_LU + 2.0 * H_U + H_RU) - (H_LD + 2.0 * H_D + H_RD)) / (8.0 * dy)

        q = self.q0 + self.A2 * np.tanh(dH_dx + dH_dy)
        L = self.L0 + self.b * np.tanh(dH_dx + dH_dy)
        return q, L

    def _apply_saltation(self, h, q_matrix, L_matrix):
        """第 1 ステップ: 跳躍（サルテーション）運動。柵はブロックのみ行う。

        右端 nx を超えて跳んだ砂は x 方向の周期境界により左端へ巻き込む。
        """
        nx, ny = self.nx, self.ny
        fence_x, fence_height = self.fence_x, self.fence_height
        hh = h.copy()

        for j in range(ny):
            for i in range(nx):
                q_requested = max(0.0, q_matrix[j, i])
                L_val = L_matrix[j, i]
                # 砂の削りすぎを制限
                available_sand = max(0.0, h[j, i] - BEDROCK)
                q_val = min(q_requested, available_sand)

                hh[j, i] -= q_val
                if q_val <= 0.0:
                    continue

                if L_val >= 0.0:
                    i_s_raw = i + int(np.round(L_val))
                    # 柵を跨ぐ場合、跳躍砂は柵の手前に叩き落とす
                    if i < fence_x <= i_s_raw:
                        if h[j, fence_x - 1] < fence_height:
                            hh[j, fence_x - 1] += q_val
                        else:
                            if fence_x < nx:
                                hh[j, fence_x] += q_val
                    else:
                        # 柵を跨がない着地は x 周期境界で巻き込む
                        hh[j, i_s_raw % nx] += q_val
                else:
                    hh[j, i] += q_val
        return hh

    def _apply_fence_adjacent_creep(self, H_next_creep, i, j, q_out):
        """柵に隣接するセルの特殊クリープ（隙間を約 9% がすり抜ける）。"""
        nx, ny = self.nx, self.ny
        if i == self.fence_x - 1:
            # 柵の手前（左側）: 右方向（2,4,7）のすり抜け合計を 0.09 に固定
            C_fence = [
                10/33,  # 0: 左上
                0/33,  # 1: 上
                1/33,  # 2: 右上 (★隙間を通り抜ける)
                10/33,  # 3: 左
                1/33,  # 4: 右   (★隙間を通り抜ける)
                10/33,  # 5: 左下
                0/33,  # 6: 下
                1/33   # 7: 右下 (★隙間を通り抜ける)  
            ]
        else:  # i == fence_x + 1
            # 柵の後方（右側）: 左方向（0,3,5）のすり抜け合計を 0.09 に固定
            C_fence = [
                1/33,  # 0: 左上 (★隙間を通り抜ける)
                0/33,  # 1: 上
                10/33,  # 2: 右上
                1/33,  # 3: 左   (★隙間を通り抜ける)
                10/33,  # 4: 右
                1/33,  # 5: 左下 (★隙間を通り抜ける)  
                0/33,  # 6: 下
                10/33   # 7: 右下
            ]

        for idx, (dx_dir, dy_dir) in enumerate(DIRECTIONS):
            q_i = C_fence[idx] * q_out
            next_x = (i + dx_dir) % nx  # x 方向は周期境界
            next_y = j + dy_dir
            if 0 <= next_y < ny:  # y 方向は壁境界
                H_next_creep[j, i] -= q_i
                H_next_creep[next_y, next_x] += q_i

    def _apply_normal_creep(self, H_current, H_next_creep, i, j, q_out):
        """通常セルの安息角モデルによるクリープ。"""
        nx, ny = self.nx, self.ny
        mu, M, g = self.mu, self.M, self.g

        F = np.zeros(8)
        for idx, (dx_dir, dy_dir) in enumerate(DIRECTIONS):
            next_x = (i + dx_dir) % nx  # x 方向は周期境界
            next_y = j + dy_dir
            if 0 <= next_y < ny:  # y 方向は壁境界
                dh = H_current[j, i] - H_current[next_y, next_x]
                theta = np.arctan(dh)
                if theta > np.arctan(mu):
                    F[idx] = M * g * (np.sin(theta) - mu * np.cos(theta))

        F_sum = np.sum(F)
        if F_sum > 0:
            C = F / F_sum
            for idx, (dx_dir, dy_dir) in enumerate(DIRECTIONS):
                q_i = C[idx] * q_out
                next_x = (i + dx_dir) % nx  # x 方向は周期境界
                next_y = j + dy_dir
                if 0 <= next_y < ny:  # y 方向は壁境界
                    H_next_creep[j, i] -= q_i
                    H_next_creep[next_y, next_x] += q_i

    def _apply_fence_core_creep(self, H_next_creep, i, j, q_out):
        """柵の真下（fence_x）セルの挙動（念のためのガード）。"""
        nx, ny = self.nx, self.ny
        C_fence_core = [3/18, 0/18, 3/18, 3/18, 3/18, 3/18, 0/18, 3/18]
        for idx, (dx_dir, dy_dir) in enumerate(DIRECTIONS):
            q_i = C_fence_core[idx] * q_out
            next_x = (i + dx_dir) % nx  # x 方向は周期境界
            next_y = j + dy_dir
            if 0 <= next_y < ny:  # y 方向は壁境界
                H_next_creep[j, i] -= q_i
                H_next_creep[next_y, next_x] += q_i

    def _apply_creep(self, H_current):
        """第 2 ステップ: 転がり（クリープ）運動。

        x 方向は周期境界のため両端の列（i=0, nx-1）も含めて全列を処理する。
        """
        nx, ny = self.nx, self.ny
        H_next_creep = H_current.copy()

        for j in range(ny):
            for i in range(nx):
                q_out = self.D * H_current[j, i]
                if q_out <= 0:
                    continue

                if i == self.fence_x:
                    self._apply_fence_core_creep(H_next_creep, i, j, q_out)
                elif i == self.fence_x - 1 or i == self.fence_x + 1:
                    self._apply_fence_adjacent_creep(H_next_creep, i, j, q_out)
                else:
                    self._apply_normal_creep(H_current, H_next_creep, i, j, q_out)
        return H_next_creep

    def _step(self):
        """1 ステップ（砂供給 → 跳躍 → クリープ）進める。"""
        # 風上（左端）から一定量の砂を供給
        self.h[:, : int(self.L0)] += 0.05
        q_matrix, L_matrix = self._calc_saltation_flux(self.h)
        hh = self._apply_saltation(self.h, q_matrix, L_matrix)
        self.h = self._apply_creep(hh)

    # ------------------------------------------------------------------
    # 実行・保存
    # ------------------------------------------------------------------
    def run(self, save_path=None, show=True, verbose=False):
        """nt_max ステップ実行し、最終ステップの描写を保存する。

        save_path : 保存先パス。None なら param_label() から自動生成。
        show      : True で Jupyter にインライン表示する。
        verbose   : True で各ステップ後に砂総量を表示。
        戻り値    : 保存した画像ファイルのパス。
        """
        for nt in range(1, self.nt_max + 1):
            self._step()
            if verbose:
                print(f"ステップ {nt} 完了 | 砂の総量: {np.sum(self.h):.2f}")

        png_path = self.save_state(nt=self.nt_max, save_path=save_path, show=show)
        html_path = self.save_surface_3d(nt=self.nt_max, show=show)
        return png_path, html_path

    def param_label(self):
        """パラメータを表す文字列（ファイル名・タイトル用）。"""
        return f"L0-{self.L0}_A2-{self.A2}_b-{self.b}_q0-{self.q0}_D-{self.D}"

    def save_state(self, nt, save_path=None, show=True):
        """現在の状態を 2D 断面図・平面図の 2 軸で描き、画像保存する。"""
        if save_path is None:
            save_path = f"dune_{self.param_label()}_step{nt}.png"

        x, y, h = self.x, self.y, self.h
        nx, ny = self.nx, self.ny
        fence_x, fence_height = self.fence_x, self.fence_height

        fig = plt.figure(figsize=(14, 5))
        fig.suptitle(f"Step {nt}  |  {self.param_label()}", fontsize=13, fontweight="bold", y=1.02)

        # 2D 断面図
        ax2 = fig.add_subplot(1, 2, 1)
        center_y_idx = ny // 2
        ax2.plot(x, h[center_y_idx, :], color="saddlebrown", linewidth=2, label="Sand Height")
        ax2.vlines(fence_x, -10, fence_height, colors="black", linestyles="solid", linewidth=3, label="Fence")
        ax2.set_title(f"Cross Section (Y = {int(y[center_y_idx])})", fontsize=12)
        ax2.set_xlabel("X Coordinate")
        ax2.set_ylabel("Height")
        ax2.set_xlim(0, nx)
        ax2.set_ylim(-12, fence_height + 20)
        ax2.grid(True, linestyle=":", alpha=0.6)
        ax2.legend(loc="upper right")

        # 2D 平面図
        ax3 = fig.add_subplot(1, 2, 2)
        X_ij, Y_ij = np.meshgrid(x, y, indexing="xy")
        sc = ax3.scatter(X_ij, Y_ij, c=h, marker="^", s=15, cmap="bwr", vmin=-10, vmax=10)
        ax3.axvline(fence_x, color="black", linestyle="--", linewidth=2)
        ax3.set_title("Top-down Height Map", fontsize=12)
        ax3.set_xlim(0, nx)
        ax3.set_ylim(0, ny)
        ax3.grid(True, linestyle=":", alpha=0.6)
        fig.colorbar(sc, ax=ax3, label="Height (h)")

        fig.tight_layout()
        fig.savefig(save_path, dpi=120, bbox_inches="tight")
        if show:
            plt.show()
        else:
            plt.close(fig)
        return save_path

    def save_surface_3d(self, nt, save_path=None, show=True):
        """現在の状態を plotly の 3D サーフェスで描き、HTML 保存する。"""
        if save_path is None:
            save_path = f"dune3d_{self.param_label()}_step{nt}.html"

        height = np.asarray(self.h, dtype=float)
        bwr_colorscale = [
            [0.0, "rgb(0,0,255)"],
            [0.5, "rgb(245,245,245)"],
            [1.0, "rgb(255,0,0)"],
        ]

        data = [go.Surface(
            x=self.x,
            y=self.y,
            z=height,
            colorscale=bwr_colorscale,
            cmin=float(np.min(height)),
            cmax=float(np.max(height)),
            colorbar=dict(title="height"),
            name="sand",
        )]

        # 柵（半透明の面 + 支柱）
        yy = np.asarray(self.y, dtype=float)
        base = float(min(-10.0, np.min(height)))
        fx = np.full((len(yy), 2), float(self.fence_x))
        fy = np.column_stack([yy, yy])
        fz = np.tile([base, float(self.fence_height)], (len(yy), 1))
        data.append(go.Surface(
            x=fx, y=fy, z=fz,
            colorscale=[[0.0, "rgb(80,80,80)"], [1.0, "rgb(80,80,80)"]],
            showscale=False,
            opacity=0.45,
            name="fence",
        ))

        n_posts = 9
        for pi in np.linspace(0, len(yy) - 1, n_posts).astype(int):
            data.append(go.Scatter3d(
                x=[float(self.fence_x), float(self.fence_x)],
                y=[yy[pi], yy[pi]],
                z=[base, float(self.fence_height)],
                mode="lines",
                line=dict(color="black", width=4),
                showlegend=False,
                hoverinfo="skip",
            ))

        fig = go.Figure(data=data)
        fig.update_layout(
            title=f"dune  |  {self.param_label()}  (step {nt})",
            width=900,
            height=650,
            margin=dict(l=0, r=0, t=40, b=0),
            scene=dict(
                xaxis_title="X",
                yaxis_title="Y",
                zaxis_title="Z",
                aspectmode="manual",
                aspectratio=dict(x=2.0, y=1.0, z=0.4),
                camera=dict(eye=dict(x=1.8, y=1.8, z=1.2)),
            ),
        )
        fig.write_html(save_path, include_plotlyjs="cdn")
        if show:
            fig.show()
        return save_path

## 定数スイープの実行

各変数について「テストしたい値のリスト」を用意し、`itertools.product` で全組み合わせを回す。

> **注意:** 組み合わせ数 = 各リスト長の積。1 ケースあたり 400 ステップ回すため、
> リストを長くすると実行時間が急増する。まずは下記のダミー値（=少数）で試すこと。
> 特定の 1 変数だけ調べたいときは、その変数のリストだけ複数値にし、他は 1 値にすると良い。

In [ ]:
import itertools
import numpy as np


def frange(start, stop, step):
    """start〜stop を step 刻みで並べたリストを返す（stop を含む・丸め誤差対策済み）。

    例: frange(2.0, 7.0, 0.1) -> [2.0, 2.1, 2.2, ..., 6.9, 7.0]（51 個）
    """
    n = int(round((stop - start) / step)) + 1
    decimals = max(0, -int(np.floor(np.log10(step))) + 2)  # 刻みに応じた丸め桁数
    return [round(start + k * step, decimals) for k in range(n)]


# --- 各変数のテスト範囲（粗く走査。計 5*4*4*5*3 = 1200 ケース） ---
L0_list = frange(1.0, 9.0, 2.0)     # [1,3,5,7,9]       跳躍距離の基準値
A2_list = frange(0.5, 2.0, 0.5)     # [0.5,1.0,1.5,2.0] 跳躍量 q の勾配依存係数
b_list  = frange(0.5, 2.0, 0.5)     # [0.5,1.0,1.5,2.0] 跳躍距離 L の勾配依存係数
q0_list = frange(1.0, 5.0, 1.0)     # [1,2,3,4,5]       跳躍量の基準値
D_list  = frange(1.0, 3.0, 1.0)     # [1,2,3]           クリープで動く砂の割合

# 全組み合わせ = 各リスト長の積
combinations = list(itertools.product(L0_list, A2_list, b_list, q0_list, D_list))
print(f"各リスト長: L0={len(L0_list)}, A2={len(A2_list)}, b={len(b_list)}, "
      f"q0={len(q0_list)}, D={len(D_list)}")
print(f"実行ケース数: {len(combinations)}")

results = []
for idx, (L0, A2, b, q0, D) in enumerate(combinations, start=1):
    print(f"[{idx}/{len(combinations)}] L0={L0}, A2={A2}, b={b}, q0={q0}, D={D} を実行中...")

    # インスタンス生成 → 400 ステップ実行 → 2D画像 & 3D(HTML) を保存（show=False で notebook 肥大化を防ぐ）
    sim = DuneSimulation(L0=L0, A2=A2, b=b, q0=q0, D=D)
    png_path, html_path = sim.run(show=False)

    results.append((png_path, html_path))
    print(f"    保存: {png_path} / {html_path}")

print("すべて完了しました。")
len(results)
